# T26 -- Lung-Region Attention Module on ViT-Base/16 (avg-pool head)

Applies T18's Lung-Region Attention module to ViT-Base across **seven arms (A0-A6)**: A0 vanilla,
A1 gate-only, A2 full, A3 multiply-gate, A4 guidance-only, A5 CBAM, A6 (A2 + background-suppression
loss). Same pipeline, split, metrics, winner rule and outputs as T23 (ResNet50), so the four
backbones are directly comparable.

**What is ViT-specific** (`src/modules/lung_attention.py::ViTLungAttention`): the 196 patch tokens
are reshaped to a 14x14 map, gated, and **average-pooled** into the head. timm's default ViT head
reads only the CLS token, which a spatial gate never reaches. Arm A0 uses the same avg-pool head, so
it is the control for A1-A6 -- it is not T17's CLS-token baseline.

**Relation to the 2026-10-01 local run** (`artifacts/vit_lung_attention/`,
`notebooks/T_26_Vit_Base_Model.ipynb`): this notebook replaces it. Deliberate protocol changes, all
to match T18/T23: batch size 32 (was 8), early stopping and best checkpoint on validation **loss**
(was validation macro-F1), attention initialised at 0.5 (was ~0.0025), spatial-only CBAM for A5
(was channel+spatial), attention metrics at 224x224 (were at 14x14), fp32 test evaluation, the
committed split manifest, and checkpoints that carry their own config. Numbers from the two runs
are therefore not interchangeable.

**How to run on Kaggle**
1. Settings: **Accelerator = GPU T4**, **Internet = On** (pretrained weights + pip).
   Add-ons -> Secrets: add `WANDB_API_KEY` and attach it to this notebook (see the W&B cell below).
2. Add Input: the *COVID-19 Radiography Database* dataset.
3. The repo is cloned from GitHub. The branch must contain `ViTLungAttention` and
   `configs/vit_base_lung_attention.yaml` -- push `t26-vit-rerun-results` (or merge it) first, or
   attach the repo folder as a Kaggle Dataset.
4. **Save Version -> Save & Run All (Commit)**. One session cannot finish everything (12 h limit).
   For each later session: Add Input -> **Your Work** -> this notebook's previous output, then Save
   Version again. Finished units are skipped; the final cell lists what is left.
5. When the last cell prints `ALL DONE`, download `/kaggle/working/T26_vit_base_lung_attention/`.
   The `.pt` checkpoints (about 330 MB each) are in `runs/<arm>/` and `runs_multiseed/` -- keep
   them, T28-T35 need them.

## S0 -- Setup

In [ ]:
import time
SESSION_START = time.time()
# Kaggle kills sessions at 12h. Stop *starting* new work after this many hours, leaving margin for
# the cheap end-of-notebook cells and for Kaggle to save outputs.
SESSION_BUDGET_H = 11.0

import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "GPU is OFF -- Settings > Accelerator > GPU T4 before running the rest of this notebook."
device = torch.device("cuda")

In [ ]:
import os, shutil
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()


def find_dirs(root, predicate, max_depth=6,
              skip=("COVID-19_Radiography_Dataset", "images", "masks", "wandb", ".git", "__pycache__")):
    # Depth-limited directory search that never descends into the 21k-image dataset folders.
    found, stack = [], [(Path(root), 0)]
    while stack:
        d, depth = stack.pop()
        try:
            if predicate(d):
                found.append(d)
                continue
            if depth < max_depth:
                stack.extend((c, depth + 1) for c in d.iterdir() if c.is_dir() and c.name not in skip)
        except OSError:
            pass
    return sorted(found)


# The repo copy must contain this task's src/ changes. Checked by file content rather than by path,
# so a stale copy (an older dataset version, an old notebook output) is never silently used.
REQUIRED_CODE = {
    "src/datasets/covid_cxr.py": "def preload_resized_cache",
    "src/modules/comparison.py": "def run_significance_tests",
    "src/modules/lung_attention.py": "class ViTLungAttention",
    "src/modules/gradcam.py": 'startswith("vit")',
    "src/modules/training.py": "experiment_tag",
    "configs/vit_base_lung_attention.yaml": "T26",
}

def repo_is_current(d):
    d = Path(d)
    return all((d / f).is_file() and marker in (d / f).read_text(encoding="utf-8") for f, marker in REQUIRED_CODE.items())


REPO_URL = "https://github.com/Ravindu-Pathirana/Chest-X-ray-Disease-Detection.git"
# Fallback only, if no up-to-date repo dataset is attached. Tried in order; the first branch whose
# checkout contains T26's src/ changes is used.
REPO_BRANCHES = ["main", "t26-vit-rerun-results"]

if ON_KAGGLE:
    # Kept outside /kaggle/working so the saved output contains results only, not a repo copy.
    REPO_ROOT = Path("/tmp/Chest-X-ray-Disease-Detection")
    if not repo_is_current(REPO_ROOT):
        _candidates = find_dirs("/kaggle/input", lambda d: (d / "src" / "modules" / "lung_attention.py").is_file())
        _current = [d for d in _candidates if repo_is_current(d)]
        print("Repo copies under /kaggle/input:", [str(c) for c in _candidates] or "none")
        shutil.rmtree(REPO_ROOT, ignore_errors=True)
        if _current:
            shutil.copytree(_current[0], REPO_ROOT, ignore=shutil.ignore_patterns(".git"))
            print("Using repo from dataset input:", _current[0])
        else:
            print("No up-to-date repo dataset attached -- falling back to git clone.")
            for REPO_BRANCH in REPO_BRANCHES:
                shutil.rmtree(REPO_ROOT, ignore_errors=True)
                !git clone --quiet --depth 1 --branch $REPO_BRANCH --single-branch $REPO_URL $REPO_ROOT
                if repo_is_current(REPO_ROOT):
                    print("Using branch:", REPO_BRANCH)
                    break
                print(f"Branch {REPO_BRANCH!r} does not contain T26's src/ changes yet.")
    assert repo_is_current(REPO_ROOT), (
        "The repo copy is missing T26's src/ changes (" + ", ".join(REQUIRED_CODE) + "). "
        "Push/merge the t26-vit-rerun-results branch, or attach your up-to-date repo folder as a Kaggle Dataset, "
        "then re-run from S0."
    )
    %pip install -q -r $REPO_ROOT/requirements.txt
    %pip install -q timm grad-cam
else:
    REPO_ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "src").is_dir())

print("REPO_ROOT:", REPO_ROOT)

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT))
from src.utils import load_config, merge_overrides, set_seed

### Restore previous sessions' results

If a previous session's results are attached as an input -- either the notebook's own output
(**Add Input -> Your Work**) or the `T26_vit_base_lung_attention` folder re-uploaded as a Kaggle
Dataset -- copy them into this session's output folder so finished units are skipped. They're found by
content (`session_timing.json` / the S7 done-marker), not by path, so the dataset's name and whether
you uploaded the folder or its contents don't matter. Attach the **latest** results: each session's
output already contains everything from the sessions before it. If several are attached, the one with
the most finished units is used.

In [ ]:
T26_ROOT = (Path("/kaggle/working/T26_vit_base_lung_attention") if ON_KAGGLE
            else REPO_ROOT / "artifacts" / "T26_vit_base_lung_attention")
T26_ROOT.mkdir(parents=True, exist_ok=True)

if ON_KAGGLE:
    def _is_t26_output(d):
        # Matched by T26-only files, not by folder name: results re-uploaded as a Kaggle Dataset are mounted
        # under the dataset's slug. (T18's artifact folder also has runs/ + sweep/, so those alone don't qualify.)
        return (d / "session_timing.json").is_file() or (d / "smoke_test" / "t26_done.json").is_file()

    _prior = find_dirs("/kaggle/input", _is_t26_output)
    if _prior:
        _best = max(_prior, key=lambda d: len(list(d.rglob("t26_done.json"))))
        if len(_prior) > 1:
            print("Several previous outputs attached:", [str(p) for p in _prior], "-> restoring the most complete one.")
        shutil.copytree(_best, T26_ROOT, dirs_exist_ok=True)
        print(f"Restored {_best} -> {T26_ROOT} ({len(list(T26_ROOT.rglob('t26_done.json')))} finished units found).")
    else:
        print("No previous T26 output attached -- starting fresh (expected for the first session).")
print("T26_ROOT:", T26_ROOT)

### Weights & Biases

Every sweep config and every arm is logged as its own W&B run (per-epoch train/val loss, accuracy,
macro-F1, AUROC, ILAR, learning rate; best-epoch and final test numbers in the run summary). Runs are
named `vit_base_patch16_224_T26-<arm>_seed<seed>` and tagged `T26`, so they filter cleanly in the
project `chest-xray-disease-classification`.

**Setup (once):** Kaggle notebook menu -> **Add-ons -> Secrets -> Add a new secret**, label
`WANDB_API_KEY`, value = your key from https://wandb.ai/authorize, and tick **Attach to notebook**.
Never paste the key into a cell.

Save Version runs have no keyboard input, so a missing key would otherwise kill the run. If the
secret is missing or the login fails, this cell switches to `WANDB_MODE=offline`: training continues,
runs are stored under `/kaggle/working/wandb/` and can be uploaded later with `wandb sync`. Every
number the notebook reports comes from its own JSON/CSV files, not from W&B.

In [ ]:
import wandb

# Set to your W&B team/entity name to log into a shared team project; None = your own account.
WANDB_ENTITY = None

try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    WANDB_ONLINE = True
    print("W&B login OK -- runs will sync live to your dashboard.")
except Exception as e:
    os.environ["WANDB_MODE"] = "offline"
    WANDB_ONLINE = False
    print(f"W&B secret/login unavailable ({type(e).__name__}: {e}) -- falling back to WANDB_MODE=offline.")
    print("To log online: Add-ons -> Secrets -> WANDB_API_KEY (attached to this notebook), then re-run.")

## S1 -- Config + resume/budget helpers

Loads `configs/vit_base_lung_attention.yaml` (arm A2's config, copied from `baseline.yaml`; other
arms via `merge_overrides()`), then defines the done-marker and session-budget helpers every
expensive section uses.

`PIPELINE_VERSION` is stamped into every done-marker. A unit only counts as finished if its marker
carries the current version, so results from an older data pipeline (e.g. a run from before the RAM
cache / per-run loader change) are retrained rather than mixed into the same ablation -- an A0
trained differently from A1-A6 would confound every delta.

In [ ]:
import json
import pandas as pd

cfg = load_config(REPO_ROOT / "configs" / "vit_base_lung_attention.yaml")
BACKBONE = cfg["model"]["name"]          # "vit_base_patch16_224"
if WANDB_ENTITY:
    cfg = merge_overrides(cfg, {"tracking.entity": WANDB_ENTITY})
print("W&B:", "online" if WANDB_ONLINE else "offline", "| project:", cfg["tracking"]["project"],
      "| entity:", cfg["tracking"]["entity"] or "(your default account)")
print("Loaded config for experiment:", cfg["experiment"]["name"])
print(json.dumps(cfg, indent=2))

In [ ]:
from contextlib import contextmanager

PIPELINE_VERSION = "t26-vit-avgpool-ramcache-v1"  # bump if data pipeline/protocol changes -> forces retraining
DONE_MARKER = "t26_done.json"
EVAL_MARKER = "t26_eval_done.json"


def _marker_ok(path):
    try:
        return json.loads(Path(path).read_text()).get("pipeline") == PIPELINE_VERSION
    except (OSError, json.JSONDecodeError):
        return False

def is_done(d):
    return _marker_ok(Path(d) / DONE_MARKER)

def read_done(d):
    return json.loads((Path(d) / DONE_MARKER).read_text())

def mark_done(d, **info):
    Path(d).mkdir(parents=True, exist_ok=True)
    (Path(d) / DONE_MARKER).write_text(json.dumps({"pipeline": PIPELINE_VERSION, **info}, indent=2, default=str))


# --- Session budget: durations of finished units are persisted, so later sessions plan with real numbers ---
TIMING_PATH = T26_ROOT / "session_timing.json"
_timing = json.loads(TIMING_PATH.read_text()) if TIMING_PATH.exists() else {}
DEFAULT_EST_H = {"arm": 2.0, "sweep_cfg": 0.5, "eval_arm": 0.3}  # conservative until measured

def hours_left():
    return SESSION_BUDGET_H - (time.time() - SESSION_START) / 3600

def can_start(kind, label=""):
    seen = _timing.get(kind, [])
    est_h = max(seen) / 3600 * 1.15 if seen else DEFAULT_EST_H[kind]
    if hours_left() > est_h:
        return True
    print(f"[budget] Not starting {kind} {label}: {hours_left():.2f}h left of {SESSION_BUDGET_H}h, needs ~{est_h:.2f}h. "
          f"It runs next session.")
    return False

@contextmanager
def timed(kind):
    t0 = time.time()
    yield
    _timing.setdefault(kind, []).append(time.time() - t0)
    TIMING_PATH.write_text(json.dumps(_timing, indent=2))
    print(f"[timing] {kind} took {(time.time() - t0) / 60:.1f} min; {hours_left():.2f}h left in this session.")

## S2 -- Data layer -- RAM-cached, mask-paired dataset + verification

Same pipeline and same committed split manifest as T18 (`artifacts/splits/split_manifest_v1.csv`), so
ViT-Base and DenseNet121 are compared on identical images. `cache_in_ram=True` decodes + resizes every
image/mask once (JointTransform's own first step); the verification cell below asserts the cached
tensors are bit-identical to the on-disk path, in both eval and train (augmented) mode.

Every training run gets a **fresh** train loader seeded with that run's seed (`fresh_train_loader`).
A shared loader's generator carries state between runs, so arm N's data order would depend on how
many epochs ran before it in the same session -- and would silently change once runs are split
across sessions.

In [ ]:
import numpy as np
import torch.nn as nn
from src.datasets import (
    build_dataloaders, compute_class_weights, make_train_loader,
    CachedCXRWithMaskDataset, CXRWithMaskDataset, JointTransform,
)

_KAGGLE_DATA_DIR = "/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset"
if ON_KAGGLE:
    DATA_DIR = _KAGGLE_DATA_DIR if Path(_KAGGLE_DATA_DIR).is_dir() else str(find_dirs(
        "/kaggle/input", lambda d: d.name == "COVID-19_Radiography_Dataset", skip=("images", "masks", ".git"))[0])
else:
    DATA_DIR = str(next(p for p in (
        REPO_ROOT.parent.parent / "Baseline Model" / "data" / "COVID-19_Radiography_Dataset",
        Path("/Volumes/My Disk 2/My Projects/Chest Disease Detection/COVID-19_Radiography_Dataset"),
    ) if p.is_dir()))
print("Using DATA_DIR:", DATA_DIR)

SPLIT_MANIFEST = REPO_ROOT / "artifacts" / "splits" / "split_manifest_v1.csv"
assert SPLIT_MANIFEST.exists(), f"{SPLIT_MANIFEST} not found -- it must be committed (T18 generated it)."

# Workers share the RAM cache copy-on-write under fork (Linux/Kaggle). Under Windows' spawn each
# worker would pickle its own 2 GB copy, so run single-process locally.
NUM_WORKERS = 4 if ON_KAGGLE else 0
SEED = cfg["experiment"]["seed"]

_t0 = time.time()
train_loader, val_loader, test_loader, class_names, train_targets, datasets = build_dataloaders(
    DATA_DIR,
    img_size=cfg["dataset"]["image_size"],
    batch_size=cfg["training"]["batch_size"],
    seed=SEED,
    num_workers=NUM_WORKERS,
    split_manifest_path=SPLIT_MANIFEST,
    cache_in_ram=True,
    persistent_workers=True,
)
CACHE_IMAGES, CACHE_MASKS = datasets["train"].images, datasets["train"].masks
print(f"Cached {len(CACHE_IMAGES)} image/mask pairs in RAM in {time.time() - _t0:.0f}s "
      f"({(CACHE_IMAGES.nbytes + CACHE_MASKS.nbytes) / 1e9:.2f} GB)")
print(f"Classes ({len(class_names)}): {class_names}")
print(f"Train/Val/Test sizes: {len(datasets['train'])}/{len(datasets['val'])}/{len(datasets['test'])}")

class_weights = compute_class_weights(train_targets, num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print("Class weights:", class_weights.cpu().tolist())


def fresh_train_loader(seed):
    return make_train_loader(datasets["train"], cfg["training"]["batch_size"], seed,
                             num_workers=NUM_WORKERS, persistent_workers=True)

def cached_subset(base_indices, train):
    return CachedCXRWithMaskDataset(datasets["train"].base_dataset, np.asarray(base_indices),
                                    JointTransform(img_size=224, train=train), CACHE_IMAGES, CACHE_MASKS)

In [ ]:
# --- S2 verification: split sizes, batch contract, and RAM cache == disk (bit-exact) ---
import random

CLASSES = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]
assert (len(datasets["train"]), len(datasets["val"]), len(datasets["test"])) == (14815, 3175, 3175)
assert class_names == CLASSES, class_names
print("[1] split sizes + class order OK")

imgs, labels, masks = next(iter(train_loader))
assert imgs.shape[1:] == (3, 224, 224), imgs.shape
assert masks.shape[1:] == (1, 224, 224), masks.shape
assert set(torch.unique(masks).tolist()) <= {0.0, 1.0}
print("[2] batch shapes OK:", imgs.shape, masks.shape)
del train_loader  # training uses fresh_train_loader(seed) per run; release this loader's workers

_probe = np.asarray(datasets["train"].indices[:12])
for _train_mode in (False, True):
    _disk = CXRWithMaskDataset(datasets["train"].base_dataset, _probe, JointTransform(img_size=224, train=_train_mode))
    _ram = cached_subset(_probe, train=_train_mode)
    for k in range(len(_probe)):
        random.seed(k); torch.manual_seed(k); a = _disk[k]
        random.seed(k); torch.manual_seed(k); b = _ram[k]
        assert a[1] == b[1] and torch.equal(a[0], b[0]) and torch.equal(a[2], b[2]), (
            f"RAM cache differs from disk at probe sample {k} (train={_train_mode})")
print("[3] RAM cache is bit-identical to the on-disk pipeline (eval + augmented train mode)")

### Alignment eyeball check

Confirm masks still line up after augmentation.

In [ ]:
import matplotlib.pyplot as plt
from src.datasets import IMAGENET_MEAN, IMAGENET_STD

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for k, ax in enumerate(axes.flat):
    show = (imgs[k] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(show)
    ax.imshow(masks[k, 0].numpy(), cmap="Reds", alpha=0.35)
    ax.set_title(class_names[labels[k]])
    ax.axis("off")
plt.suptitle("S2 alignment check -- augmented train images + masks")
plt.tight_layout()
plt.show()

## S3 -- The Lung-Region Attention module (reused) + the ViT wrapper

Canonical code: `src/modules/lung_attention.py`. `LungRegionAttention` is unchanged from T18;
`build_model(backbone_name="vit_...")` returns `ViTLungAttention`, covered by
`tests/test_vit_lung_attention.py`.

In [ ]:
from src.modules import (
    build_model, freeze_backbone, unfreeze_final_blocks, print_trainable_parameters,
    LungRegionAttention, LogitsOnly,
)
import timm

## S4 -- Assemble the model on ViT-Base + parity checks

- `gate_mode="none"` must give bit-identical logits to arm A0.
- A0 must equal `head(mean(patch tokens))` of plain timm ViT, and must **differ** from timm's own
  CLS-token forward (the head the gate could never influence).
- Parameter counts are checked against the values verified locally. CPU-only, under a minute.

In [ ]:
torch.manual_seed(0)
_a0 = build_model(num_classes=4, use_attention=False, pretrained=False, backbone_name=BACKBONE)
_wrapped = build_model(num_classes=4, use_attention=True, gate_mode="none", pretrained=False, backbone_name=BACKBONE)
_plain = timm.create_model(BACKBONE, pretrained=False, num_classes=4)
_wrapped.backbone.load_state_dict(_a0.backbone.state_dict())
_plain.load_state_dict(_a0.backbone.state_dict())
_plain.eval(); _wrapped.eval(); _a0.eval()

_x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    _a0_logits, _a0_att, _ = _a0(_x)
    _wrapped_logits, _att, _ = _wrapped(_x)
    _manual = _plain.head(_plain.forward_features(_x)[:, 1:].mean(dim=1))
    _cls_logits = _plain(_x)

assert _a0_att is None and _att.shape == (2, 1, 14, 14), _att.shape
torch.testing.assert_close(_wrapped_logits, _a0_logits, rtol=0, atol=0)
print("PARITY 1 PASSED: gate_mode='none' logits are bit-identical to arm A0")
torch.testing.assert_close(_a0_logits, _manual, rtol=0, atol=1e-6)
print("PARITY 2 PASSED: A0 == head(mean(patch tokens)) of plain timm ViT (avg-pool head)")
assert not torch.allclose(_a0_logits, _cls_logits, atol=1e-4)
print("PARITY 3 PASSED: A0 differs from timm's CLS-token forward, as intended")

_vanilla_params = sum(p.numel() for p in _plain.parameters())
_a0_params = sum(p.numel() for p in _a0.parameters())
print(f"A0 params: {_a0_params:,} | vanilla timm ViT-Base params: {_vanilla_params:,}")
assert _a0_params == _vanilla_params == 85_801_732, (_a0_params, _vanilla_params)

In [ ]:
_attn = LungRegionAttention(768, reduction=cfg["module"]["reduction"], gate_mode="residual")
_attn_params = sum(p.numel() for p in _attn.parameters())
print(f"Attention module params: {_attn_params:,} ({100.0 * _attn_params / _vanilla_params:.4f}% of vanilla ViT-Base)")
assert _attn_params == 73_921, _attn_params

model = build_model(num_classes=4, use_attention=True, gate_mode="residual", pretrained=False, backbone_name=BACKBONE)
freeze_backbone(model)
print_trainable_parameters(model, "phase1 ")
assert sum(p.numel() for p in model.parameters() if p.requires_grad) == 76_997  # head 3,076 + attn 73,921

unfreeze_final_blocks(model, num_blocks=cfg["training"]["unfreeze_blocks"])
print_trainable_parameters(model, "phase2 ")
assert sum(p.numel() for p in model.parameters() if p.requires_grad) == 14_254_277  # + blocks 10-11 + final norm
_names = [n for n, p in model.named_parameters() if p.requires_grad]
assert any(n.startswith("backbone.blocks.10.") for n in _names) and any(n.startswith("backbone.blocks.11.") for n in _names)
assert not any(n.startswith("backbone.blocks.9.") for n in _names)
del model, _wrapped, _plain, _a0
print("\nS4 verification passed.")

## S5 -- Metrics (reused) + ViT's Grad-CAM taps

Both taps are the wrapper's own identity modules (`pre_attn`: the 14x14 patch-token map entering
the gate; `post_attn`: the map entering the avg-pool head), so plain Grad-CAM applies with no token
reshape. All attention metrics upsample the 14x14 map to 224x224 first, exactly as for the CNNs' 7x7
maps.

In [ ]:
from src.modules import (
    ilar, attention_iou, attention_dice, attention_entropy, background_attention,
    energy_inside_lung, cam_for, get_taps,
)

_mask = torch.zeros(1, 1, 224, 224); _mask[:, :, 64:160, 48:176] = 1.0
_frac = _mask.mean().item()
assert abs(ilar(_mask.clone(), _mask).item() - 1.0) < 1e-4
assert abs(ilar(torch.ones(1, 1, 7, 7), _mask).item() - _frac) < 0.02
assert ilar(1.0 - _mask, _mask).item() < 1e-4
assert abs(attention_iou(_mask.clone(), _mask).item() - 1.0) < 1e-4
assert abs(attention_dice(_mask.clone(), _mask).item() - 1.0) < 1e-4
assert abs(attention_entropy(torch.full((1, 1, 7, 7), 0.5)).item() - 0.69315) < 1e-4
assert background_attention(_mask.clone(), _mask).item() < 1e-4
assert abs(energy_inside_lung(_mask.clone(), _mask).item() - 1.0) < 1e-4
print("All metric self-checks passed.")

# Arm A0 has no gate, so EIL at the post-gate and pre-gate taps must agree.
_demo_model = build_model(num_classes=4, use_attention=False, pretrained=False, backbone_name=BACKBONE)
_demo_images = torch.randn(2, 3, 224, 224)
_tap_post, _tap_pre = get_taps(_demo_model, backbone_name=BACKBONE)
_cam_post, _ = cam_for(_demo_model, _demo_images, _tap_post, torch.device("cpu"))
_cam_pre, _ = cam_for(_demo_model, _demo_images, _tap_pre, torch.device("cpu"))
_eil_post = energy_inside_lung(_cam_post, _mask.repeat(2, 1, 1, 1))
_eil_pre = energy_inside_lung(_cam_pre, _mask.repeat(2, 1, 1, 1))
assert torch.allclose(_eil_post, _eil_pre, atol=1e-3), (_eil_post, _eil_pre)
assert _tap_pre is _demo_model.pre_attn and _tap_post is _demo_model.post_attn
del _demo_model
print("Grad-CAM harness self-check passed on ViT-Base's taps (pre_attn / post_attn, 14x14 maps).")

## S6 -- Training loop (reused) + resumable run helpers

`run_full_arm` / `train_phase` / `evaluate` are unchanged from T18. The two helpers below wrap them
with: skip-if-done, the session budget guard, a fresh per-run train loader, and a done-marker
recording the arm's settings (used later to label the comparison table without reloading checkpoints).

In [ ]:
import gc
from src.modules import run_full_arm, train_phase, evaluate, build_optimizer, build_scheduler, best_history_row
from src.utils import initialize_wandb, finish_run, generate_run_name

RUNS_ROOT = T26_ROOT / "runs"
SWEEP_ROOT = T26_ROOT / "sweep"
FIG_DIR = T26_ROOT / "figures"
for _d in (RUNS_ROOT, SWEEP_ROOT, FIG_DIR):
    _d.mkdir(parents=True, exist_ok=True)


def train_arm_if_needed(arm_name, arm_cfg, out_dir):
    # Returns the arm's phase-2 test results, or None if it didn't fit in this session's budget.
    out_dir = Path(out_dir)
    if is_done(out_dir):
        print(f"[resume] {arm_name}: already trained -- loading saved test results.")
        return json.loads((out_dir / "phase2_finetune_test_results.json").read_text())
    if not can_start("arm", arm_name):
        return None
    (out_dir / EVAL_MARKER).unlink(missing_ok=True)  # any earlier evaluation belongs to a different checkpoint
    m = arm_cfg["module"]
    seed = arm_cfg["experiment"]["seed"]
    loader = fresh_train_loader(seed)
    try:
        with timed("arm"):
            results = run_full_arm(
                arm_name, arm_cfg, loader, val_loader, test_loader, class_names, criterion,
                device, out_dir, backbone_name=BACKBONE, wandb_enabled=True, experiment_tag="T26",
            )
    finally:
        del loader
        gc.collect(); torch.cuda.empty_cache()
    mark_done(out_dir, arm=arm_name, seed=seed, use_attention=m["use_attention"], attention=m.get("attention", "lung"),
              gate_mode=m.get("gate_mode", "residual"), lambda_att=m["lambda_att"], lambda_bg=m.get("lambda_bg", 0.0))
    return results


TUNE_PHASE1_EPOCHS = 4
TUNE_PHASE2_EPOCHS = 6
TUNE_PATIENCE = 3
TOLERANCE = 0.005  # 0.5 percentage points of val macro-F1, absolute


def run_sweep_config_if_needed(name, lambda_att, lambda_bg):
    # Short-schedule, validation-only sweep point. Returns its result row, or None if deferred by the budget.
    cfg_dir = SWEEP_ROOT / name
    if is_done(cfg_dir):
        print(f"[resume] sweep {name}: already done.")
        return read_done(cfg_dir)["row"]
    if not can_start("sweep_cfg", name):
        return None
    set_seed(SEED)
    cfg_dir.mkdir(parents=True, exist_ok=True)
    loader = fresh_train_loader(SEED)
    t = cfg["training"]
    try:
        initialize_wandb(cfg, run_name=generate_run_name(BACKBONE, f"T26-sweep-{name}", SEED),
                         tags=["T26", BACKBONE, "sweep", f"seed{SEED}"])
        with timed("sweep_cfg"):
            sweep_model = build_model(
                num_classes=cfg["model"]["num_classes"], use_attention=True, gate_mode=cfg["module"]["gate_mode"],
                pretrained=cfg["model"]["pretrained"], backbone_name=BACKBONE, drop_rate=cfg["model"]["drop_rate"],
            ).to(device)
            freeze_backbone(sweep_model)
            opt1 = build_optimizer(sweep_model, t["optimizer"], t["phase1_lr"], t["weight_decay"])
            sweep_model = train_phase(
                sweep_model, loader, val_loader, criterion, opt1,
                build_scheduler(opt1, cfg["scheduler"]["name"], TUNE_PHASE1_EPOCHS), device,
                epochs=TUNE_PHASE1_EPOCHS, patience=TUNE_PATIENCE, phase_name="phase1_frozen",
                output_dir=cfg_dir, lambda_att=lambda_att, lambda_bg=lambda_bg, wandb_enabled=True,
            )
            # No evaluate() call -- test data stays out of hyperparameter selection entirely.
            unfreeze_final_blocks(sweep_model, t["unfreeze_blocks"])
            opt2 = build_optimizer(sweep_model, t["optimizer"], t["phase2_lr"], t["weight_decay"])
            sweep_model = train_phase(
                sweep_model, loader, val_loader, criterion, opt2,
                build_scheduler(opt2, cfg["scheduler"]["name"], TUNE_PHASE2_EPOCHS), device,
                epochs=TUNE_PHASE2_EPOCHS, patience=TUNE_PATIENCE, phase_name="phase2_finetune",
                output_dir=cfg_dir, lambda_att=lambda_att, lambda_bg=lambda_bg, wandb_enabled=True,
            )
        best = best_history_row(cfg_dir / "phase2_finetune_history.json")
        row = {
            "name": name, "lambda_att": lambda_att, "lambda_bg": lambda_bg,
            "phase1_lr": t["phase1_lr"], "phase2_lr": t["phase2_lr"], "weight_decay": t["weight_decay"],
            "optimizer": t["optimizer"], "scheduler": cfg["scheduler"]["name"],
            "best_epoch": best["epoch"], "best_val_loss": best["val_loss"], "val_acc_at_best_loss": best["val_acc"],
            "val_macro_f1_at_best": best["val_f1"], "val_ilar_at_best": best["val_ilar"],
        }
    finally:
        finish_run()
        del loader
        gc.collect(); torch.cuda.empty_cache()
    mark_done(cfg_dir, row=row)
    return row


def select_largest_within_tolerance(df, lam_col, reference_name):
    # Pre-registered rule: largest lambda whose val macro-F1 is within TOLERANCE of the lambda=0
    # reference; ties broken by higher val ILAR.
    ref_f1 = df.loc[df["name"] == reference_name, "val_macro_f1_at_best"].iloc[0]
    cands = df[df["val_macro_f1_at_best"] >= ref_f1 - TOLERANCE]
    if cands.empty:
        raise RuntimeError(f"No {lam_col} within 0.5pp of {reference_name} -- extend the grid downward (e.g. 0.05, 0.03).")
    top = cands[cands[lam_col] == cands[lam_col].max()].sort_values("val_ilar_at_best", ascending=False).iloc[0]
    return top, float(ref_f1)


def plot_sweep(df, lam_col, title, path):
    d = df.sort_values(lam_col)
    fig, ax1 = plt.subplots(figsize=(8, 5))
    ax2 = ax1.twinx()
    ax1.plot(d[lam_col], d["val_macro_f1_at_best"], "o-", color="tab:blue")
    ax2.plot(d[lam_col], d["val_ilar_at_best"], "s-", color="tab:red")
    ax1.set_xlabel(lam_col); ax1.set_ylabel("val macro-F1", color="tab:blue"); ax2.set_ylabel("val ILAR", color="tab:red")
    plt.title(title); fig.tight_layout(); plt.savefig(path, dpi=150); plt.show()

## S7 -- Smoke test + overfit test (ViT-Base) -- first session only

The same three checks T18/T23 ran before their first expensive GPU run, skipped automatically once
they have passed. Checks 2 and 3 use the **pretrained** backbone at lr=1e-4: a ViT trained from
scratch at lr=1e-3 reached only 31% on the 32-image overfit check (measured locally), so T23's
settings would fail here for reasons unrelated to the wiring being tested. The pretrained settings
have not been run before, so only by-construction facts are hard assertions (finite losses,
att_loss starts at log 2, att_loss decreases, ILAR rises with lambda, better than chance);
T23's magnitude thresholds are printed as NOTEs instead of stopping the session.

In [ ]:
RUN_S7 = not is_done(T26_ROOT / "smoke_test")
print("S7 checks:", "running" if RUN_S7 else "already passed in an earlier session -- skipping")

In [ ]:
if RUN_S7:
    # --- Check 1: smoke run (200 train / 100 val real images, 1+1 epochs) ---
    _smoke_out = T26_ROOT / "smoke_test"
    _rng = np.random.default_rng(0)
    _smoke_train_loader = torch.utils.data.DataLoader(
        cached_subset(_rng.choice(datasets["train"].indices, size=200, replace=False), train=True), batch_size=8, shuffle=True)
    _smoke_val_loader = torch.utils.data.DataLoader(
        cached_subset(_rng.choice(datasets["val"].indices, size=100, replace=False), train=False), batch_size=8, shuffle=False)

    _smoke_model = build_model(num_classes=4, use_attention=True, gate_mode="residual", pretrained=False, backbone_name=BACKBONE).to(device)
    freeze_backbone(_smoke_model)
    _opt1 = torch.optim.AdamW([p for p in _smoke_model.parameters() if p.requires_grad], lr=1e-3)
    _smoke_model = train_phase(_smoke_model, _smoke_train_loader, _smoke_val_loader, criterion, _opt1, scheduler=None,
                               device=device, epochs=1, patience=5, phase_name="phase1_frozen",
                               output_dir=_smoke_out, lambda_att=0.5, wandb_enabled=False)
    unfreeze_final_blocks(_smoke_model, cfg["training"]["unfreeze_blocks"])
    _opt2 = torch.optim.AdamW([p for p in _smoke_model.parameters() if p.requires_grad], lr=1e-5)
    _smoke_model = train_phase(_smoke_model, _smoke_train_loader, _smoke_val_loader, criterion, _opt2, scheduler=None,
                               device=device, epochs=1, patience=5, phase_name="phase2_finetune",
                               output_dir=_smoke_out, lambda_att=0.5, wandb_enabled=False)
    for _phase in ("phase1_frozen", "phase2_finetune"):
        _hist = json.loads((_smoke_out / f"{_phase}_history.json").read_text())
        assert all(np.isfinite(row["train_loss"]) for row in _hist), f"non-finite loss in {_phase}"
    evaluate(_smoke_model, _smoke_val_loader, class_names, device, _smoke_out, "phase2_finetune")
    assert (_smoke_out / "phase2_finetune_summary_metrics.csv").exists()
    del _smoke_model
    print("\nCHECK 1 PASSED: no crash, finite losses, history + evaluate() outputs written.")
else:
    print('S7 check 1 skipped (passed in an earlier session).')

In [ ]:
if RUN_S7:
    # --- Check 2: overfit test (32 real images, no augmentation, 100 steps, ALL params unfrozen) ---
    from src.modules import compute_total_loss

    _targets = np.array(datasets["train"].base_dataset.targets)
    _rng2 = np.random.default_rng(1)
    _overfit_idx = []
    for _c in range(4):
        _class_train_idx = datasets["train"].indices[_targets[datasets["train"].indices] == _c]
        _overfit_idx.extend(_rng2.choice(_class_train_idx, size=8, replace=False))
    _of_images, _of_labels, _of_masks = next(iter(torch.utils.data.DataLoader(cached_subset(_overfit_idx, train=False), batch_size=32)))
    _of_images, _of_labels, _of_masks = _of_images.to(device), _of_labels.to(device), _of_masks.to(device)

    _of_model = build_model(num_classes=4, use_attention=True, gate_mode="residual", pretrained=True, backbone_name=BACKBONE).to(device)
    for p in _of_model.parameters():
        p.requires_grad = True
    _of_criterion = nn.CrossEntropyLoss()
    _of_optimizer = torch.optim.AdamW(_of_model.parameters(), lr=0.0001)

    _of_model.train()
    _att_loss_history, _acc_history = [], []
    for _step in range(100):
        _of_optimizer.zero_grad(set_to_none=True)
        _class_logits, _att, _att_logits = _of_model(_of_images)
        _loss, _cls_loss, _att_loss, _bg_loss = compute_total_loss(
            _class_logits, _att_logits, _of_labels, _of_masks, _of_criterion, lambda_att=0.5, target_mode="soft")
        _loss.backward()
        _of_optimizer.step()
        _att_loss_history.append(_att_loss.item())
        _acc_history.append((_class_logits.argmax(dim=1) == _of_labels).float().mean().item())
        if _step in (0, 9, 24, 49, 74, 99):
            print(f"step {_step+1:3d}: loss={_loss.item():.4f} att={_att_loss.item():.4f} train_acc={_acc_history[-1]:.4f}")

    assert _acc_history[-1] > 0.5, "wiring bug -- 32 images not learned at all (chance is 0.25)"
    if _acc_history[-1] < 1.0:
        print(f"NOTE: overfit accuracy {_acc_history[-1]:.3f} < 1.0 after 100 steps (threshold not calibrated for ViT).")
    assert abs(_att_loss_history[0] - 0.6931) < 0.05, "att_loss did not start near log(2) -- check zero-init"
    assert _att_loss_history[-1] < _att_loss_history[0], "att_loss did not decrease -- check lambda/logits wiring"
    if _att_loss_history[-1] > _att_loss_history[0] - 0.2:
        print(f"NOTE: att_loss fell only {_att_loss_history[0] - _att_loss_history[-1]:.3f} in 100 steps (T23 expected > 0.2).")
    assert _att_loss_history[-1] > 0.05, "att_loss went to ~0 -- target is being thresholded, not soft"
    del _of_model
    print("\nCHECK 2 PASSED.")
else:
    print('S7 check 2 skipped (passed in an earlier session).')

In [ ]:
if RUN_S7:
    # --- Check 3: lambda-sensitivity spot check (50 steps at lambda=0 vs lambda=5) ---
    _rng3 = np.random.default_rng(2)
    _lam_idx = []
    for _c in range(4):
        _class_train_idx = datasets["train"].indices[_targets[datasets["train"].indices] == _c]
        _lam_idx.extend(_rng3.choice(_class_train_idx, size=8, replace=False))
    _lam_images, _lam_labels, _lam_masks = next(iter(torch.utils.data.DataLoader(cached_subset(_lam_idx, train=False), batch_size=32)))
    _lam_images, _lam_labels, _lam_masks = _lam_images.to(device), _lam_labels.to(device), _lam_masks.to(device)

    def _run_lambda(lam, seed=7, steps=50):
        torch.manual_seed(seed)
        m = build_model(num_classes=4, use_attention=True, gate_mode="residual", pretrained=True, backbone_name=BACKBONE).to(device)
        for p in m.parameters():
            p.requires_grad = True
        crit = nn.CrossEntropyLoss()
        opt = torch.optim.AdamW(m.parameters(), lr=0.0001)
        m.train()
        for _ in range(steps):
            opt.zero_grad(set_to_none=True)
            cls_logits, att, att_logits = m(_lam_images)
            loss, *_ = compute_total_loss(cls_logits, att_logits, _lam_labels, _lam_masks, crit, lambda_att=lam, target_mode="soft")
            loss.backward()
            opt.step()
        m.eval()
        with torch.no_grad():
            _, att_final, _ = m(_lam_images)
            out = ilar(att_final.float(), _lam_masks).mean().item()
        del m, opt
        gc.collect(); torch.cuda.empty_cache()
        return out

    _ilar_lam0 = _run_lambda(0.0)
    _ilar_lam5 = _run_lambda(5.0)
    print(f"ILAR: lambda=0 -> {_ilar_lam0:.4f} | lambda=5 -> {_ilar_lam5:.4f}")
    assert _ilar_lam5 > _ilar_lam0, "guidance signal is not reaching the module -- ILAR did not rise with lambda"
    if _ilar_lam5 < _ilar_lam0 + 0.05:
        print(f"NOTE: ILAR rose by only {_ilar_lam5 - _ilar_lam0:.3f} in 50 steps (T23 expected > 0.05).")
    torch.cuda.empty_cache()
    mark_done(T26_ROOT / "smoke_test", checks=["smoke_run", "overfit", "lambda_sensitivity"])
    print("\nALL S7 CHECKS PASSED -- recorded; later sessions skip them.")
else:
    print('S7 check 3 skipped (passed in an earlier session).')

## S7b -- Config audit

Confirms the protocol values this notebook is supposed to run with (see the header of
`configs/vit_base_lung_attention.yaml`). `phase1_lr`, `phase2_lr` and `unfreeze_blocks` are the T26
owner's values from the 2026-10-01 local run; there is no ViT HPO sweep to audit against, unlike
T23's check against T14/T15.

In [ ]:
assert BACKBONE == "vit_base_patch16_224"
assert cfg["model"]["drop_rate"] == 0.0
assert cfg["module"]["reduction"] == 8
assert cfg["training"]["optimizer"] == "adamw"
assert cfg["training"]["phase1_lr"] == 1e-3
assert cfg["training"]["phase2_lr"] == 1e-5
assert cfg["training"]["weight_decay"] == 1e-4
assert cfg["scheduler"]["name"] == "cosine"
assert cfg["training"]["unfreeze_blocks"] == 2
assert cfg["training"]["batch_size"] == 32          # aligned with T18/T23 (the local run used 8)
assert cfg["checkpoint"]["monitor"] == "val_loss"   # aligned with T18/T23 (the local run used val macro-F1)
print("S7b PASSED: config matches the T26 protocol.")

## S8 -- Arm A0 -- the vanilla avg-pool control (full schedule)

`use_attention=False`. Sanity gate against the two existing ViT-Base numbers: T17's CLS-token
baseline (macro-F1 0.9225) and the 2026-10-01 avg-pool A0 (0.9299, batch 8).

In [ ]:
a0_cfg = merge_overrides(cfg, {
    "module.use_attention": False, "module.lambda_att": 0.0, "module.lambda_bg": 0.0,
    "experiment.name": "T26-A0-vanilla",
})
A0_OUT = RUNS_ROOT / "A0_vanilla"
a0_results = train_arm_if_needed("A0_vanilla", a0_cfg, A0_OUT)

In [ ]:
if a0_results is None:
    print("A0 not trained yet (deferred to the next session).")
else:
    macro_f1 = a0_results["classification_report"]["macro avg"]["f1-score"]
    print(f"A0 test macro-F1: {macro_f1:.4f}")
    if 0.905 <= macro_f1 <= 0.950:
        print("Sanity gate: PASS -- in the expected 90.5-95.0% range (T17 CLS-token 0.9225; earlier avg-pool A0 0.9299).")
    elif 0.88 <= macro_f1 < 0.905:
        print("Sanity gate: BORDERLINE -- check S7b's config values actually took effect.")
    elif macro_f1 < 0.80:
        print("Sanity gate: FAIL -- likely a pipeline bug. Stop and debug before trusting later arms.")
    else:
        print("Sanity gate: outside the pre-registered bands -- inspect manually.")

## S9 -- lambda_att sweep (short schedule) + pre-registered selection

Re-derived for ViT-Base on this protocol -- neither the CNNs' `lambda_att*` nor the earlier local
run's value is assumed to transfer. 5 configs x (4+6) epochs, validation only. Each config is its
own resumable unit; selection runs once all 5 exist. Rule (same as T18/T23): the **largest**
`lambda_att` whose validation macro-F1 is within 0.5 pp of the `lambda_att=0` reference.

In [ ]:
LAMBDA_ATT_GRID = [0.0, 0.1, 0.3, 0.5, 1.0]
att_rows = [run_sweep_config_if_needed(f"lam_att_{lam}", lambda_att=lam, lambda_bg=0.0) for lam in LAMBDA_ATT_GRID]
SWEEP_ATT_COMPLETE = all(r is not None for r in att_rows)
print(f"lambda_att sweep: {sum(r is not None for r in att_rows)}/{len(att_rows)} configs done.")

In [ ]:
LAMBDA_ATT_STAR = None
if SWEEP_ATT_COMPLETE:
    tuning_df = pd.DataFrame(att_rows)
    tuning_df.to_csv(SWEEP_ROOT / "tuning_summary.csv", index=False)
    display(tuning_df)
    plot_sweep(tuning_df, "lambda_att", "T26 (ViT-Base) lambda_att sweep", FIG_DIR / "lambda_att_sweep.png")

    top, ref_f1 = select_largest_within_tolerance(tuning_df, "lambda_att", "lam_att_0.0")
    lambda_att_selected = {
        "name": top["name"], "lambda_att": float(top["lambda_att"]),
        "phase1_lr": float(top["phase1_lr"]), "phase2_lr": float(top["phase2_lr"]),
        "weight_decay": float(top["weight_decay"]), "optimizer": top["optimizer"], "scheduler": top["scheduler"],
        "gate_mode": cfg["module"]["gate_mode"], "target_mode": cfg["module"]["target_mode"],
        "reduction": cfg["module"]["reduction"],
        "rule": "largest lambda_att whose val macro-F1 is within 0.5pp of lam_att_0.0; ties -> higher val ILAR",
        "reference_row": "lam_att_0.0", "reference_val_macro_f1": ref_f1,
        "schedule": f"short({TUNE_PHASE1_EPOCHS}+{TUNE_PHASE2_EPOCHS})", "seed": SEED, "backbone": BACKBONE,
        "pipeline": PIPELINE_VERSION,
    }
    (SWEEP_ROOT / "selected_config.json").write_text(json.dumps(lambda_att_selected, indent=2))
    LAMBDA_ATT_STAR = lambda_att_selected["lambda_att"]
    print(f"Selected lambda_att* = {LAMBDA_ATT_STAR}")
else:
    print("lambda_att sweep incomplete -- selection (and everything that needs lambda_att*) waits for a later session.")

## S10 -- Full-schedule runs -- A2, A1, A5, A4, A3 (seed 42)

Priority order as in T18: A2 (headline) -> A1 -> A5 (CBAM comparator) -> A4 -> A3. `run_full_arm`
re-seeds at the start of every arm, and each arm gets its own fresh train loader.

In [ ]:
ARM_CONFIGS = {}
if LAMBDA_ATT_STAR is not None:
    _lung = {"module.use_attention": True, "module.attention": "lung", "module.lambda_bg": 0.0}
    ARM_CONFIGS = {
        "A2_full": merge_overrides(cfg, {**_lung, "module.gate_mode": "residual", "module.lambda_att": LAMBDA_ATT_STAR,
                                         "experiment.name": "T26-A2-full"}),
        "A1_gate_only": merge_overrides(cfg, {**_lung, "module.gate_mode": "residual", "module.lambda_att": 0.0,
                                              "experiment.name": "T26-A1-gate-only"}),
        "A5_cbam": merge_overrides(cfg, {"module.use_attention": True, "module.attention": "cbam",
                                         "module.lambda_att": 0.0, "module.lambda_bg": 0.0,  # CBAM is never mask-supervised
                                         "experiment.name": "T26-A5-cbam"}),
        "A4_guidance_only": merge_overrides(cfg, {**_lung, "module.gate_mode": "none", "module.lambda_att": LAMBDA_ATT_STAR,
                                                  "experiment.name": "T26-A4-guidance-only"}),
        "A3_multiply": merge_overrides(cfg, {**_lung, "module.gate_mode": "multiply", "module.lambda_att": LAMBDA_ATT_STAR,
                                             "experiment.name": "T26-A3-multiply"}),
    }
    for arm_name, arm_cfg in ARM_CONFIGS.items():
        train_arm_if_needed(arm_name, arm_cfg, RUNS_ROOT / arm_name)
else:
    print("Waiting on S9's lambda_att* -- A1-A5 run in a later session.")

## S11 -- lambda_bg sweep (short schedule) + pre-registered selection (design doc section 5b)

Independent second sweep: `lambda_att` fixed at `lambda_att*`, residual gate, `lambda_bg` over
`{0, 0.1, 0.3, 0.5, 1.0}`; largest value within 0.5pp of `lambda_bg=0` (plain A2), ties -> higher ILAR.

In [ ]:
LAMBDA_BG_GRID = [0.0, 0.1, 0.3, 0.5, 1.0]
bg_rows, SWEEP_BG_COMPLETE = [], False
if LAMBDA_ATT_STAR is not None:
    bg_rows = [run_sweep_config_if_needed(f"lam_bg_{lam}", lambda_att=LAMBDA_ATT_STAR, lambda_bg=lam) for lam in LAMBDA_BG_GRID]
    SWEEP_BG_COMPLETE = all(r is not None for r in bg_rows)
    print(f"lambda_bg sweep: {sum(r is not None for r in bg_rows)}/{len(bg_rows)} configs done.")
else:
    print("Waiting on S9's lambda_att* -- the lambda_bg sweep runs in a later session.")

In [ ]:
LAMBDA_BG_STAR = None
if SWEEP_BG_COMPLETE:
    tuning_bg_df = pd.DataFrame(bg_rows)
    tuning_bg_df.to_csv(SWEEP_ROOT / "tuning_summary_bg.csv", index=False)
    display(tuning_bg_df)
    plot_sweep(tuning_bg_df, "lambda_bg", f"T26 (ViT-Base) lambda_bg sweep (lambda_att*={LAMBDA_ATT_STAR})",
               FIG_DIR / "lambda_bg_sweep.png")

    top, ref_f1 = select_largest_within_tolerance(tuning_bg_df, "lambda_bg", "lam_bg_0.0")
    lambda_bg_selected = {
        "name": top["name"], "lambda_bg": float(top["lambda_bg"]), "lambda_att": float(LAMBDA_ATT_STAR),
        "gate_mode": "residual", "phase1_lr": float(top["phase1_lr"]), "phase2_lr": float(top["phase2_lr"]),
        "weight_decay": float(top["weight_decay"]), "optimizer": top["optimizer"], "scheduler": top["scheduler"],
        "target_mode": cfg["module"]["target_mode"], "reduction": cfg["module"]["reduction"],
        "rule": "largest lambda_bg whose val macro-F1 is within 0.5pp of lam_bg_0.0 (plain A2); ties -> higher val ILAR",
        "reference_row": "lam_bg_0.0", "reference_val_macro_f1": ref_f1,
        "schedule": f"short({TUNE_PHASE1_EPOCHS}+{TUNE_PHASE2_EPOCHS})", "seed": SEED, "backbone": BACKBONE,
        "pipeline": PIPELINE_VERSION,
    }
    (SWEEP_ROOT / "selected_bg_config.json").write_text(json.dumps(lambda_bg_selected, indent=2))
    LAMBDA_BG_STAR = lambda_bg_selected["lambda_bg"]
    print(f"Selected lambda_bg* = {LAMBDA_BG_STAR}")
else:
    print("lambda_bg sweep incomplete -- A6 waits for a later session.")

## S12 -- Arm A6 (A2 + background suppression, full schedule)

Uses `lambda_att*` (S9) and `lambda_bg*` (S11) with the residual gate -- the seventh and final arm.

In [ ]:
a6_cfg = None
if LAMBDA_ATT_STAR is not None and LAMBDA_BG_STAR is not None:
    a6_cfg = merge_overrides(cfg, {
        "module.use_attention": True, "module.attention": "lung", "module.gate_mode": "residual",
        "module.lambda_att": LAMBDA_ATT_STAR, "module.lambda_bg": LAMBDA_BG_STAR,
        "experiment.name": "T26-A6-full-bg",
    })
    train_arm_if_needed("A6_full_bg", a6_cfg, RUNS_ROOT / "A6_full_bg")
else:
    print("Waiting on both sweeps -- A6 runs in a later session.")

ALL_ARMS = ["A0_vanilla", "A1_gate_only", "A2_full", "A3_multiply", "A4_guidance_only", "A5_cbam", "A6_full_bg"]
ARM_DIRS = {a: RUNS_ROOT / a for a in ALL_ARMS}
_missing = [a for a in ALL_ARMS if not is_done(ARM_DIRS[a])]
ALL_ARMS_DONE = not _missing
print("All 7 arms trained (acceptance criterion A5)." if ALL_ARMS_DONE else f"Arms still to train: {_missing}")
if a6_cfg is not None:
    ALL_ARM_CONFIGS = {"A0_vanilla": a0_cfg, **ARM_CONFIGS, "A6_full_bg": a6_cfg}

## S13 -- Evaluation, faithfulness, comparison table (all 7 arms)

Runs once all 7 arms are trained. Per arm: per-image predictions + Grad-CAM EIL at both taps on a
fixed stratified 1,000-image test subset (same images for every arm), each arm a resumable unit.
The comparison table has T18's schema plus a `lambda_bg` column (otherwise A6 and A2 rows look
identical).

In [ ]:
from src.modules import stratified_cam_subset, build_per_image_predictions, build_comparison_table, check_acceptance_criteria

EVAL_READY = False
CAM_N = 1000


def load_arm_model(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    m = ckpt["config"]["module"]
    model = build_model(
        num_classes=ckpt["config"]["model"]["num_classes"], use_attention=m["use_attention"],
        attention=m.get("attention", "lung"), gate_mode=m.get("gate_mode", "residual"), reduction=m.get("reduction", 8),
        backbone_name=ckpt["config"]["model"]["name"], pretrained=False,
        drop_rate=ckpt["config"]["model"].get("drop_rate", 0.0),
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    return model.eval(), ckpt.get("class_names", class_names)


def evaluate_arm_if_needed(arm_name, arm_dir):
    # Returns an arms_for_table entry, or None if deferred by the budget.
    arm_dir = Path(arm_dir)
    marker = arm_dir / EVAL_MARKER
    fresh = _marker_ok(marker) and json.loads(marker.read_text()).get("cam_n") == len(cam_subset)
    if fresh:
        per_image_df = pd.read_csv(arm_dir / "per_image_predictions.csv")
    else:
        if not can_start("eval_arm", arm_name):
            return None
        with timed("eval_arm"):
            model, arm_class_names = load_arm_model(arm_dir / f"{BACKBONE}_{arm_name}.pt")
            per_image_df = build_per_image_predictions(
                model, datasets["test"], arm_class_names, device, cam_subset=cam_subset, backbone_name=BACKBONE)
            per_image_df.to_csv(arm_dir / "per_image_predictions.csv", index=False)
            del model; gc.collect(); torch.cuda.empty_cache()
        marker.write_text(json.dumps({"pipeline": PIPELINE_VERSION, "cam_n": len(cam_subset)}))

    info = read_done(arm_dir)
    evaluate_results = json.loads((arm_dir / "phase2_finetune_test_results.json").read_text())
    summary = {
        "attention_ilar": evaluate_results.get("attention_ilar"),
        "attention_dice": evaluate_results.get("attention_dice"),
        "attention_iou": evaluate_results.get("attention_iou"),
        "cam_eil_post_mean": float(per_image_df["eil_post"].mean()) if per_image_df["eil_post"].notna().any() else None,
        "cam_eil_pre_mean": float(per_image_df["eil_pre"].mean()) if per_image_df["eil_pre"].notna().any() else None,
        "cam_subset_n": len(cam_subset),
    }
    (arm_dir / "attention_metrics.json").write_text(json.dumps(summary, indent=2))
    gate = "none" if not info["use_attention"] else ("cbam" if info["attention"] == "cbam" else info["gate_mode"])
    return {"gate_mode": gate, "lambda_att": info["lambda_att"], "lambda_bg": info["lambda_bg"],
            "evaluate_results": evaluate_results, "per_image_df": per_image_df}


if ALL_ARMS_DONE:
    cam_subset = stratified_cam_subset(datasets["test"], n=CAM_N, seed=SEED)
    (T26_ROOT / "cam_subset_indices.json").write_text(
        json.dumps({"n": len(cam_subset), "seed": SEED, "positions": cam_subset.tolist()}, indent=2))
    print(f"Grad-CAM subsample: {len(cam_subset)} / {len(datasets['test'])} test images")

    arms_for_table = {}
    for arm_name in ALL_ARMS:
        entry = evaluate_arm_if_needed(arm_name, ARM_DIRS[arm_name])
        if entry is not None:
            arms_for_table[arm_name] = entry
    EVAL_READY = len(arms_for_table) == len(ALL_ARMS)
    print(f"{len(arms_for_table)}/{len(ALL_ARMS)} arms evaluated.")
else:
    print("Evaluation waits until all 7 arms are trained.")

In [ ]:
if EVAL_READY:
    comparison_df = build_comparison_table(arms_for_table, reference_arm="A0_vanilla")
    comparison_df.insert(3, "lambda_bg", [arms_for_table[a]["lambda_bg"] for a in comparison_df["arm"]])
    comparison_df.to_csv(T26_ROOT / "T26_comparison_table.csv", index=False)
    display(comparison_df)

    # A1-A4 checked against A2 for direct comparability with T18; S14 picks the actual winner.
    verdicts = check_acceptance_criteria(comparison_df, headline_arm="A2_full")
    for name, v in verdicts.items():
        print(f"[{'PASS' if v['pass'] else 'FAIL'}] {name}: {v['value']:.4f} (target: {v['target']})")
    (T26_ROOT / "acceptance_criteria_A1_A4.json").write_text(json.dumps(verdicts, indent=2))
    print("\nA failing verdict is a legitimate finding to report, not a reason to keep tuning.")
else:
    print("Comparison table waits for evaluation of all 7 arms.")

## S14 -- Winner-selection rule (design doc section 8) -- across all 7 arms

1. Candidate winner iff McNemar p >= 0.05 vs. A0 (no detectable accuracy cost) **and** Wilcoxon
   p < 0.05 on paired EIL_post (a real evidence-relocation effect).
2. Rank candidates by mean EIL_post gain; efficiency (S15) only breaks ties.
3. Reported independently of DenseNet121 -- A2 is not assumed to win, nor A6 to beat A2.

In [ ]:
VIT_WINNER = None
if EVAL_READY:
    from src.modules import run_significance_tests

    sig_results = run_significance_tests(arms_for_table, reference_arm="A0_vanilla",
                                         output_json=T26_ROOT / "significance_tests_vs_A0.json")
    sig_df = pd.DataFrame(sig_results).T.rename_axis("arm").reset_index()
    display(sig_df)

    candidates = sig_df[sig_df["is_candidate_winner"] == True]
    if candidates.empty:
        print("No arm passes both gates -- a legitimate finding: report sig_df as-is, do not force a winner.")
    else:
        ranked = candidates.sort_values("mean_eil_post_gain", ascending=False)
        print(ranked[["arm", "mean_eil_post_gain", "mcnemar_p", "wilcoxon_p"]].to_string(index=False))
        VIT_WINNER = ranked.iloc[0]["arm"]
        print(f"\nViT-Base's winner (independent of DenseNet121's A2): {VIT_WINNER}")

    (T26_ROOT / "winner_selection.json").write_text(json.dumps({
        "reference_arm": "A0_vanilla",
        "candidates": list(candidates["arm"]),
        "winner": VIT_WINNER,
        "rule": "candidate iff McNemar p>=0.05 AND Wilcoxon p<0.05 on EIL_post vs. A0; rank by mean EIL_post gain; "
                "efficiency breaks ties only.",
    }, indent=2))
else:
    print("Winner selection waits for evaluation of all 7 arms.")

## S15 -- Efficiency measurement (acceptance criterion A6)

Architecture-only (no trained weights), so it runs in the first session and is skipped afterwards.
fvcore does not count the fused scaled-dot-product attention op and prints a warning about it; that
affects the absolute GFLOPs of both rows equally, so the with/without **delta** (what A6 tests) is
unaffected. Do not quote the absolute ViT GFLOPs from this table.

In [ ]:
from src.modules import check_module_efficiency

T26_EFFICIENCY_CSV = T26_ROOT / "T26_efficiency.csv"
A6_JSON = T26_ROOT / "acceptance_criteria_A6.json"

if A6_JSON.exists():
    efficiency_verdicts = json.loads(A6_JSON.read_text())
    print("[resume] efficiency already measured.")
else:
    %pip install -q fvcore
    if str(REPO_ROOT / "notebooks") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "notebooks"))
    from efficiency import benchmark_model

    benchmark_model(LogitsOnly(build_model(use_attention=False, pretrained=False, backbone_name=BACKBONE)),
                    "ViT-Base", "without_module", output_csv=T26_EFFICIENCY_CSV)
    benchmark_model(LogitsOnly(build_model(use_attention=True, pretrained=False, backbone_name=BACKBONE)),
                    "ViT-Base", "with_module", output_csv=T26_EFFICIENCY_CSV)
    efficiency_verdicts = check_module_efficiency(pd.read_csv(T26_EFFICIENCY_CSV))
    if np.isfinite(efficiency_verdicts["A6_module_is_cheap"]["gflops_delta_pct"]):
        A6_JSON.write_text(json.dumps(efficiency_verdicts, indent=2))
    else:
        # fvcore missing/failed -> GFLOPs is NaN and the verdict would be a false FAIL (this happened in T18).
        T26_EFFICIENCY_CSV.unlink(missing_ok=True)
        print("WARNING: GFLOPs could not be measured (see the [FLOPs warning] above). Verdict NOT saved; "
              "it is re-measured next session.")

_v6 = efficiency_verdicts["A6_module_is_cheap"]
print(f"[{'PASS' if _v6['pass'] else 'FAIL'}] A6_module_is_cheap (target: {_v6['target']})")
print(f"  params: +{_v6['params_delta_absolute']:,} ({_v6['params_delta_pct']:.4f}%)")
print(f"  gflops: +{_v6['gflops_delta_absolute']:.6f} ({_v6['gflops_delta_pct']:.4f}%)")

## S16 -- Figures (acceptance criterion A8)

Heat-map overlays (up to 3 per class) comparing A0 vs. A2 (T18's convention) + the A2 attention
grid. The lambda-sweep plots were saved in S9/S11.

In [ ]:
if EVAL_READY:
    import torch.nn.functional as TF_f
    from src.modules import select_heatmap_images, plot_heatmap_row, plot_attention_grid

    a0_pi = arms_for_table["A0_vanilla"]["per_image_df"]
    a2_pi = arms_for_table["A2_full"]["per_image_df"]
    heatmap_selection = select_heatmap_images(a0_pi, a2_pi, class_names, n_per_class=3, seed=SEED)

    a0_model, _ = load_arm_model(ARM_DIRS["A0_vanilla"] / f"{BACKBONE}_A0_vanilla.pt")
    a2_model, _ = load_arm_model(ARM_DIRS["A2_full"] / f"{BACKBONE}_A2_full.pt")
    a0_tap_post, _ = get_taps(a0_model, backbone_name=BACKBONE)
    a2_tap_post, _ = get_taps(a2_model, backbone_name=BACKBONE)

    base_ds = datasets["test"].base_dataset
    path_to_test_idx = {str(base_ds.samples[datasets["test"].indices[i]][0]): i for i in range(len(datasets["test"]))}

    def _gray(img_t):
        return (img_t * std + mean).clamp(0, 1).mean(0).numpy()

    def _a2_attention(img_t):
        with torch.no_grad():
            _, att, _ = a2_model(img_t.unsqueeze(0).to(device))
        return TF_f.interpolate(att.float(), size=(224, 224), mode="bilinear", align_corners=False)[0, 0].cpu().numpy()

    HEATMAP_DIR = FIG_DIR / "heatmaps"
    for _, row in heatmap_selection.iterrows():
        img_t, _label, mask_t = datasets["test"][path_to_test_idx[row["image_path"]]]
        batch = img_t.unsqueeze(0)
        a0_cam, _ = cam_for(a0_model, batch, a0_tap_post, device)
        a2_cam, _ = cam_for(a2_model, batch, a2_tap_post, device)
        plot_heatmap_row(
            image=_gray(img_t), mask=mask_t[0].numpy(), a0_cam=a0_cam[0, 0].cpu().numpy(),
            a2_attention=_a2_attention(img_t), a2_cam=a2_cam[0, 0].cpu().numpy(),
            title=f"{row['class_name']} -- {row['reason']}",
            output_path=HEATMAP_DIR / f"{row['class_name']}_{row['reason']}.png".replace(" ", "_"),
        )
    print(f"Saved {len(heatmap_selection)} heat-map overlay rows to {HEATMAP_DIR}")

    _grid_images, _grid_atts, _grid_labels = [], [], []
    for cname in class_names:
        cls_rows = a2_pi[a2_pi["true_label"] == cname]
        for _, row in cls_rows.sample(min(4, len(cls_rows)), random_state=SEED).iterrows():
            img_t, _label, _mask = datasets["test"][path_to_test_idx[row["image_path"]]]
            _grid_images.append(_gray(img_t)); _grid_atts.append(_a2_attention(img_t)); _grid_labels.append(cname)
    plot_attention_grid(_grid_images, _grid_atts, _grid_labels, class_names, output_path=FIG_DIR / "attention_grid.png")
    del a0_model, a2_model; gc.collect(); torch.cuda.empty_cache()
    print(f"Saved attention grid to {FIG_DIR / 'attention_grid.png'}")
else:
    print("Figures wait for evaluation of all 7 arms.")

## S17 -- Multi-seed repeat (design doc step 8)

Only if S14 found a clear winner: A0 and the winner at seeds 123 and 2026, full schedule, then the
same evaluation. Each run and each evaluation is resumable. With no winner (or `RUN_MULTISEED =
False`), an explicit single-seed statement is recorded instead -- never while repeats are merely
pending.

In [ ]:
RUN_MULTISEED = True
MULTISEED_SEEDS = [123, 2026]
MULTISEED_ROOT = T26_ROOT / "runs_multiseed"
MULTISEED_SUMMARY_PATH = MULTISEED_ROOT / "multiseed_summary.json"
multiseed_summary = None

if not EVAL_READY:
    print("Multi-seed repeat is decided after evaluation of all 7 arms.")
elif not RUN_MULTISEED or VIT_WINNER is None:
    _reason = "no arm passed the winner-selection rule (S14)" if VIT_WINNER is None else "RUN_MULTISEED = False"
    MULTISEED_ROOT.mkdir(parents=True, exist_ok=True)
    MULTISEED_SUMMARY_PATH.write_text(json.dumps({
        "multiseed_repeat_run": False, "seed": SEED, "reason": _reason,
        "statement": ("All T26 (ViT-Base) results are single-seed (42) point estimates; the design doc's step-8 "
                      f"multi-seed repeat was not run because {_reason}. No mean +/- std is reported."),
    }, indent=2))
    print(f"Single-seed statement recorded ({_reason}).")
else:
    _seed_dirs = {}
    for seed in MULTISEED_SEEDS:
        for arm_name in ["A0_vanilla", VIT_WINNER]:
            run_name = f"{arm_name}_seed{seed}"
            seed_cfg = merge_overrides(ALL_ARM_CONFIGS[arm_name], {"experiment.seed": seed, "experiment.name": f"T26-{run_name}"})
            _seed_dirs[(arm_name, seed)] = MULTISEED_ROOT / run_name
            train_arm_if_needed(run_name, seed_cfg, MULTISEED_ROOT / run_name)

    if all(is_done(d) for d in _seed_dirs.values()):
        per_seed_comparison_dfs = {SEED: comparison_df}
        for seed in MULTISEED_SEEDS:
            seed_table = {}
            for arm_name in ["A0_vanilla", VIT_WINNER]:
                entry = evaluate_arm_if_needed(f"{arm_name}_seed{seed}", _seed_dirs[(arm_name, seed)])
                if entry is not None:
                    seed_table[arm_name] = entry
            if len(seed_table) == 2:
                per_seed_comparison_dfs[seed] = build_comparison_table(
                    seed_table, reference_arm="A0_vanilla", output_csv=MULTISEED_ROOT / f"T26_comparison_table_seed{seed}.csv")

        if len(per_seed_comparison_dfs) == 1 + len(MULTISEED_SEEDS):
            from src.modules import summarize_multiseed
            multiseed_summary = summarize_multiseed(per_seed_comparison_dfs, arms=["A0_vanilla", VIT_WINNER],
                                                    metrics=["test_macro_f1", "EIL_post"])
            for arm_name, metrics in multiseed_summary.items():
                for metric, s in metrics.items():
                    print(f"{arm_name} {metric}: {s['mean']:.4f} +/- {s['std']:.4f} (seeds {s['seeds']})")
            MULTISEED_SUMMARY_PATH.write_text(json.dumps(multiseed_summary, indent=2))
        else:
            print("Multi-seed evaluation incomplete -- continues next session.")
    else:
        print("Multi-seed training incomplete -- continues next session.")

## S18 -- Package and hand off

Generates a first-draft `T26_module_card.md` from the computed results -- review and edit by hand
before treating it as final, as with T18's card.

In [ ]:
if EVAL_READY:
    def _table_md(df):
        try:
            return df.to_markdown(index=False)
        except ImportError:  # tabulate not installed
            return "```\n" + df.to_string(index=False) + "\n```"

    lines = [
        "# T26 -- Lung-Region Attention Module on ViT-Base -- Module Card\n",
        "**Backbone:** ViT-Base/16, avg-pool head (timm `vit_base_patch16_224`, ImageNet-pretrained). **Draft auto-generated by S18 -- review before treating as final.**\n",
        "## 1. Selected hyperparameters\n",
        f"- `lambda_att*` = {LAMBDA_ATT_STAR} (S9; {lambda_att_selected['rule']})",
        f"- `lambda_bg*` = {LAMBDA_BG_STAR} (S11; {lambda_bg_selected['rule']})",
        f"- `reduction` = {cfg['module']['reduction']} -> +73,921 params (+0.086% of ViT-Base's 85,801,732)\n",
        "- Protocol: batch 32, early stopping on val_loss, attention init 0.5, spatial-only CBAM (A5), "
        "avg-pool head for every arm including A0. Not comparable with the 2026-10-01 local run.\n",
        "## 2. Comparison table (all 7 arms, seed 42)\n",
        _table_md(comparison_df), "",
        "## 3. Acceptance criteria A1-A4 (headline arm: A2_full)\n",
        *[f"- **[{'PASS' if v['pass'] else 'FAIL'}]** {k}: {v['value']:.4f} (target: {v['target']})" for k, v in verdicts.items()],
        "",
        "## 4. Acceptance criterion A6 (module cost)\n",
        f"- **[{'PASS' if _v6['pass'] else 'FAIL'}]** params +{_v6['params_delta_absolute']:,} ({_v6['params_delta_pct']:.4f}%), "
        f"GFLOPs +{_v6['gflops_delta_absolute']:.6f} ({_v6['gflops_delta_pct']:.4f}%)\n",
        "## 5. Winner selection (design doc section 8)\n",
        f"Candidates (McNemar p>=0.05 AND Wilcoxon p<0.05 vs. A0): {list(candidates['arm']) or '(none)'}",
        f"**ViT-Base's winner: {VIT_WINNER or 'none -- no arm passed both gates, see significance_tests_vs_A0.json'}**\n",
        "## 6. Multi-seed repeat\n",
        (json.dumps(multiseed_summary, indent=2) if multiseed_summary is not None
         else (json.loads(MULTISEED_SUMMARY_PATH.read_text()).get("statement", "pending")
               if MULTISEED_SUMMARY_PATH.exists() else "pending")),
        "",
        "## 7. Reproducibility notes\n",
        f"- Data pipeline `{PIPELINE_VERSION}`: RAM-cached inputs (bit-identical to disk), a fresh seeded train "
        "loader per run, so each arm is reproducible regardless of which session it ran in.",
        "- Checkpoints (`.pt`) are not committed -- record their Kaggle Dataset / Drive URLs here.",
    ]
    (T26_ROOT / "T26_module_card.md").write_text("\n".join(lines), encoding="utf-8")
    print(f"Draft module card written to {T26_ROOT / 'T26_module_card.md'}")
else:
    print("Module card waits for evaluation of all 7 arms.")

## Progress report

What's finished, what's left, and what to do next.

In [ ]:
def _eval_done(d):
    return _marker_ok(Path(d) / EVAL_MARKER)

units = [("S7 checks", is_done(T26_ROOT / "smoke_test"))]
units += [(f"sweep lam_att_{lam}", is_done(SWEEP_ROOT / f"lam_att_{lam}")) for lam in LAMBDA_ATT_GRID]
units += [(f"train {a}", is_done(ARM_DIRS[a])) for a in ALL_ARMS]
units += [(f"sweep lam_bg_{lam}", is_done(SWEEP_ROOT / f"lam_bg_{lam}")) for lam in LAMBDA_BG_GRID]
units += [(f"evaluate {a}", _eval_done(ARM_DIRS[a])) for a in ALL_ARMS]
units += [("efficiency (A6)", A6_JSON.exists())]
if EVAL_READY and RUN_MULTISEED and VIT_WINNER is not None:
    for seed in MULTISEED_SEEDS:
        for arm_name in ["A0_vanilla", VIT_WINNER]:
            d = MULTISEED_ROOT / f"{arm_name}_seed{seed}"
            units += [(f"train {arm_name} seed{seed}", is_done(d)), (f"evaluate {arm_name} seed{seed}", _eval_done(d))]
elif not EVAL_READY:
    units += [("multi-seed repeat (decided after evaluation)", False)]

remaining = [name for name, done in units if not done]
for name, done in units:
    print(f"  [{'x' if done else ' '}] {name}")
print(f"\nThis session: {(time.time() - SESSION_START) / 3600:.2f}h used. Timing log: {TIMING_PATH}")
if remaining:
    print(f"\n{len(remaining)} unit(s) remaining. To continue: attach this session's results as an input "
          f"(Your Work -> this notebook's output, or {T26_ROOT.name}/ uploaded as a Kaggle Dataset), then Save "
          f"Version again. Finished units are skipped automatically.")
else:
    print("\nALL DONE -- download /kaggle/working/T26_vit_base_lung_attention/ and review the draft module card.")